In [8]:
suppressPackageStartupMessages({
    library(ArchR) 
    library(data.table)
    library(purrr)
    library(parallel)
    library(dplyr)
    library(Matrix)
    library(gridExtra)
})
options(repr.plot.width=15, repr.plot.height=8)

In [2]:
# I/O
io = list()
io$basedir='/rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_2'
io$output.directory <- file.path(io$basedir,"ArchR")
io$plot.dir = file.path(io$basedir,'celltype_score')
dir.create(file.path(io$plot.dir), showWarnings = FALSE)
setwd(io$output.directory)

In [3]:
io$archR.directory = file.path(io$output.directory, 'Project/')

ArchRProject.filt = loadArchRProject(io$archR.directory)

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [4]:
ArchRProject.filt = suppressWarnings(addGeneScoreMatrix(ArchRProject.filt, force=TRUE))

# Calculate impute weights
ArchRProject.filt = addImputeWeights(
  ArchRProj = ArchRProject.filt,
  reducedDims = "IterativeLSI_Harmony")

ArchR logging to : ArchRLogs/ArchR-addGeneScoreMatrix-4fcf2e9e4b84-Date-2021-12-06_Time-17-03-18.log
If there is an issue, please report to github with logFile!

2021-12-06 17:03:18 : Batch Execution w/ safelapply!, 0 mins elapsed.

.createArrowGroup : Arrow Group already exists! Dropping Group from ArrowFile! This will take ~10-30 seconds!

.dropGroupsFromArrow : Initializing Temp ArrowFile

.dropGroupsFromArrow : Adding Metadata to Temp ArrowFile

.dropGroupsFromArrow : Adding SubGroups to Temp ArrowFile

.dropGroupsFromArrow : Move Temp ArrowFile to ArrowFile

rabbit_BGRGP1 .addGeneScoreMat useTSS = FALSE

2021-12-06 17:04:17 : Computing Gene Scores using distance relative to GeneBody! , 0.991 mins elapsed.

.createArrowGroup : Arrow Group already exists! Dropping Group from ArrowFile! This will take ~10-30 seconds!

.dropGroupsFromArrow : Initializing Temp ArrowFile

.dropGroupsFromArrow : Adding Metadata to Temp ArrowFile

.dropGroupsFromArrow : Adding SubGroups to Temp ArrowFile


In [5]:
# Get gene score matrix
gene_matrix = getMatrixFromProject(
  ArchRProj = ArchRProject.filt,
  useMatrix = "GeneScoreMatrix")

# Rename columns & rows
gene_names = gene_matrix@elementMetadata$name
gene_matrix = gene_matrix@assays@data$GeneScoreMatrix
rownames(gene_matrix) = gene_names

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-4fcf45ea340c-Date-2021-12-06_Time-17-27-31.log
If there is an issue, please report to github with logFile!

2021-12-06 17:29:47 : Organizing colData, 2.264 mins elapsed.

2021-12-06 17:29:47 : Organizing rowData, 2.266 mins elapsed.

2021-12-06 17:29:47 : Organizing rowRanges, 2.266 mins elapsed.

2021-12-06 17:29:47 : Organizing Assays (1 of 1), 2.266 mins elapsed.

2021-12-06 17:30:10 : Constructing SummarizedExperiment, 2.657 mins elapsed.

2021-12-06 17:30:11 : Finished Matrix Creation, 2.67 mins elapsed.



In [6]:
plot_celltype_score = function(markernr){
    # Load markers
    markers = fread("/rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC/RNA/markers.csv", header=TRUE)[,-1]
    markers = head(markers, markernr)

    # get clusters
    meta = ArchRProject.filt@cellColData['Clusters']

    # Calculate celltype scores
    celltype_scores = lapply(colnames(markers), function(x){
        genes = markers[[x]]
        if(length(rownames(gene_matrix)[rownames(gene_matrix) %in% genes]) > 1){
            imputed_matrix = suppressMessages(imputeMatrix(mat = gene_matrix[rownames(gene_matrix) %in% genes,], 
                                     imputeWeights = getImputeWeights(ArchRProject.filt)))

            v1 = scale(t(imputed_matrix))
            score = Matrix::rowSums(v1)
            return(score)
            }
        else{
            score = rep(0, dim(gene_matrix)[2])
            return(score)
        }
    })

    # rename columns
    celltype_scores = t(do.call(rbind.data.frame, celltype_scores))
    colnames(celltype_scores) = gsub(' ', '_', colnames(markers))
    colnames(celltype_scores) = gsub('/', '_', colnames(celltype_scores))
    colnames(celltype_scores) = gsub('-', '_', colnames(celltype_scores))
    rownames(celltype_scores) = colnames(gene_matrix)

    # plot
    options(repr.plot.width=15, repr.plot.height=4)

    per_cluster = list()
    per_celltype = list()

    for(i in 1:length(colnames(celltype_scores))){
        plot = as.data.frame(celltype_scores[,i])
        colnames(plot) = 'celltype'
        plot = merge(plot, meta, by=0)

    per_celltype[[i]] = ggplot(as.data.frame(plot), aes(Clusters, celltype, fill = Clusters)) + 
                    scale_x_discrete(labels = paste0('C', 1:length(unique(plot$Clusters)))) +
                    geom_violin() +
                    geom_boxplot(fill='white', width=0.2) + 
                    ylab('Celltype Score') + 
                    theme_bw() + theme(legend.position='none') +
                    ggtitle(colnames(celltype_scores)[i]) 
    }


    meta = as.data.frame(meta)
    meta$cell = rownames(meta)
    
    for(i in 1:length(unique(meta$Clusters))){
        cells = meta[meta$Clusters==unique(meta$Clusters)[i], ]$cell
        scores_percelltype = celltype_scores[cells, ]
        scores_percelltype = reshape2::melt(scores_percelltype)


        per_cluster[[i]] = ggplot(scores_percelltype, aes(Var2, value, fill=Var2)) + 
            geom_violin(width=1.5) + 
            geom_boxplot(fill='white', width=0.15) + 
            xlab('') + ylab('Celltype Score') + 
            ggtitle(unique(meta$Clusters)[i]) + 
            theme_bw() + theme(legend.position='none', axis.text.x=element_text(angle=-90, hjust=0))
    }

    # Save
    outfile <- sprintf("%s/per_cluster_%s_markers.pdf",io$plotdir, markernr)
    pdf(outfile, width=15, height=5)
        print(per_cluster)
    dev.off()

    outfile <- sprintf("%s/per_celltype_%s_markers.pdf",io$plotdir, markernr)
    pdf(outfile, width=7, height=5)
        print(per_celltype)
    dev.off()
    
    write.csv(celltype_scores, file.path(io$plotdir,'celltype_score.csv'))
}

In [7]:
sapply(c(5,10,20,50,100), plot_celltype_score)

[[1]]


Warning message:
“position_dodge requires non-overlapping x intervals”



[[2]]


Warning message:
“position_dodge requires non-overlapping x intervals”



[[3]]


...

Warning message:
“position_dodge requires non-overlapping x intervals”



[[25]]


Warning message:
“position_dodge requires non-overlapping x intervals”



[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]

[[8]]

[[9]]

[[10]]

[[11]]

[[12]]

[[13]]

[[14]]

[[15]]

[[16]]

[[17]]

[[18]]

[[19]]

[[20]]

[[21]]

[[22]]

[[23]]

[[24]]

[[25]]

[[26]]

[[27]]

[[28]]

[[29]]

[[30]]

[[31]]

[[32]]

[[33]]

[[34]]

[[35]]

[[36]]

[[37]]

[[38]]

[[39]]

[[40]]

[[41]]

[[42]]

[[43]]

[[44]]

[[45]]

[[46]]

[[47]]

[[48]]

[[49]]

[[50]]

[[51]]

[[52]]

[[53]]

[[54]]

[[55]]

[[56]]

[[57]]

[[58]]

[[59]]

[[60]]

[[61]]



ERROR: Error in if (file == "") file <- stdout() else if (is.character(file)) {: argument is of length zero
